In [ ]:
# ── COLAB SETUP — run this cell first ──────────────────────────────────────
# Installs the stack this notebook needs. Safe to re-run.
# Colab already ships numpy / scipy / matplotlib / requests, so only the
# Qiskit packages are installed here (upgrading the rest causes conflicts).
%pip install -q --upgrade "qiskit>=1.2" "qiskit-aer>=0.15"

import importlib, sys
for mod in ("qiskit", "qiskit_aer", "numpy", "scipy", "matplotlib", "requests"):
    try:
        m = importlib.import_module(mod)
        print(f"  {mod:<12} {getattr(m, '__version__', 'ok')}")
    except Exception as exc:
        print(f"  {mod:<12} MISSING -> {exc}")

print("\nSetup check done.")
print("If a package above is MISSING, use Runtime -> Restart session, then Run all.")


# Challenge 3: Large-Scale QAOA on Native Heavy-Hex Hardware
**Track:** Advanced  
**Challenge ID:** `challenge_advanced`  
**Theme:** Combinatorial Optimization, QAOA, Max-Cut, IBM Heavy-Hex Embedding, and Scaling up to 156 Qubits!  

---

### Welcome to the Advanced Challenge!

Combinatorial optimization problems — such as graph partitioning, portfolio balancing, logistics routing, and circuit placement — are ubiquitous across science and industry. Many belong to the class of **NP-hard** problems, where classical brute-force search becomes intractable as the number of variables grows.

The **Quantum Approximate Optimization Algorithm (QAOA)** is a polynomial-depth variational algorithm designed to approximate the ground state of an Ising spin glass Hamiltonian:
$$\hat{H}_C = \sum_{(i,j) \in E} w_{ij} \, Z_i Z_j$$

In this advanced challenge, inspired by the Basque Quantum (BasQ) combinatorial optimization track, you will solve the **Maximum Cut (Max-Cut)** problem. You will start with a small graph to master QAOA mechanics, and then scale up to **hardware-native heavy-hex graph topologies capable of executing on up to 156 qubits on real IBM Quantum processors (such as IBM Heron and Eagle)**!

---

## Scaling to 156 Qubits on Real IBM Quantum Hardware

A central question in modern quantum computing is:
> *Are NISQ algorithms truly scalable to 100+ qubits on physical processors?*

**YES!** This challenge is specifically designed for 156-qubit execution on IBM Quantum hardware:

```
[Arbitrary Graph]           Dense all-to-all connectivity     --> Hundreds of SWAPs (Fails on NISQ)
[Heavy-Hex Native Graph]    Matches IBM Heron chip topology  --> ZERO SWAP GATES (Executes on 156 Qubits!)
```

1. **The SWAP Bottleneck:** If you run QAOA on a random, dense graph, the compiler must insert hundreds of SWAP gates to route two-qubit interactions across physical hardware wires. On physical qubits, noise ruins the result.
2. **Native Heavy-Hex Embedding:** The latest IBM Quantum processors (**IBM Heron at 156 qubits** and **IBM Eagle at 127 qubits**) feature a **heavy-hexagonal lattice** where each vertex has degree at most 3.
3. If your problem graph is embedded as a **native subgraph of the heavy-hex coupling map**, every single edge $(i, j)$ directly connects physical hardware neighbors. **Zero SWAP gates are required**, enabling shallow, coherent QAOA execution across the entire 156-qubit chip!

---

## How Challenges Are Graded: The Unbounded Advanced Scoring System

Your solution is evaluated across four orthogonal dimensions with **no upper score limit**:

$$\text{Final Score} = \text{round}\Big( \text{Base}(N) \times \mathcal{A}_{\text{approx}} \times \mathcal{S}_{\text{swap}} \times \mathcal{D}_{\text{depth}} \Big)$$

### 1. Balanced Polynomial Base Scaling ($5,000 + 60 \times (N^2 - 25)$)
$$\text{Base}(N) = 5,000 + 60 \times (N^2 - 25)$$
- **$N = 5$ Qubits (toy graph):** 5,000 points base
- **$N = 16$ Qubits:** 18,860 points base
- **$N = 30$ Qubits:** 57,500 points base
- **$N = 156$ Qubits (Full IBM Heron QPU!):** **~1,460,000 points base!**
Ensures scaling to higher qubit counts (e.g., $N=30$ vs $N=16$) is strictly rewarded, while keeping balance so that a high-cut, zero-SWAP $N=28$ embedding beats an unoptimized $N=30$ graph with routing overhead.

### 2. Approximation Ratio Factor ($\mathcal{A}_{\text{approx}}$)
In Max-Cut, flipping a fair coin for each node cuts 50% of the edges on average ($\alpha_{\text{random}} = 0.5$).
- If your approximation ratio $\alpha = \frac{C_{\text{qaoa}}}{C_{\text{optimal}}} \le 0.5$, your score is **0**.
- Above random guessing ($0.5 < \alpha \le 1.0$), your factor is:
$$\mathcal{A}_{\text{approx}} = \left( \frac{\alpha - 0.5}{0.5} \right)^{1.5}$$
Reaching an optimal cut ($\alpha = 1.0$) yields $\mathcal{A}_{\text{approx}} = 1.0$.

### 3. SWAP Penalty ($\mathcal{S}_{\text{swap}} = \frac{1}{1 + 0.20 \times N_{\text{swap}}}$)
Punishes routing overhead. Embedding your problem graph into the native hardware coupling map achieves $N_{\text{swap}} = 0$, earning the full $1.0\times$ multiplier!

### 4. Depth Efficiency ($\mathcal{D}_{\text{depth}} = \frac{D_{\text{ref}}}{\max(D_{\text{actual}}, D_{\text{ref}})}$)
On a heavy-hex lattice of degree $\le 3$, an optimal depth-$1$ QAOA circuit has reference gate depth $D_{\text{ref}} = 6$ (including initial $H$ superposition and edge interactions). Efficiently scheduled circuits achieve full depth efficiency.

### The 5 Pillars to Maximize Your Challenge Score
1. **Scale Qubits ($N \uparrow$):** Base potential scales quadratically ($5,000 + 60(N^2 - 25)$). Scaling from 5 up to 16, 30, or even 156 qubits on IBM hardware unlocks million-point ceilings.
2. **Error Mitigation (Dynamical Decoupling / Pauli Twirling / TREX):** Suppress idle qubit decoherence and readout misclassification to boost the measured approximation ratio ($\alpha$).
3. **Circuit Depth Optimization ($D \to 6$):** Parallelize commuting two-qubit $ZZ$ cost gates into edge matchings (edge colorings) so all two-qubit gates execute in minimal layers.
4. **Eliminate SWAP Gates ($N_{\text{swap}} = 0$):** Non-native routing triggers heavy $1 / (1 + 0.20 \times N_{\text{swap}})$ penalties. Embed your problem instance directly on native hardware edges.
5. **Hardware-Aware Layout (Heavy-Hex Subgraphs):** Design graphs that are direct subgraphs of the IBM Heavy-Hex lattice, ensuring zero SWAP routing overhead.


In [ ]:
# Run once if dependencies are needed
# %pip install -q qiskit qiskit-aer matplotlib requests scipy numpy

import math
import requests
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile, qasm2
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.transpiler import CouplingMap
from scipy.optimize import minimize

print("Qiskit SDK and scientific packages loaded successfully.")


## Part 1: The Max-Cut Problem & Cost Hamiltonian

Given an undirected graph $G = (V, E)$, the goal of **Max-Cut** is to partition the vertices $V$ into two subsets $S_0$ and $S_1$ such that the number of edges crossing the cut is maximized.

Assigning binary variables $x_i \in \{0, 1\}$ to each node, the cut value is:
$$C(x) = \sum_{(i,j) \in E} \left( x_i(1 - x_j) + x_j(1 - x_i) \right)$$

Substituting $x_i = \frac{1 - Z_i}{2}$ translates this into the quantum **cost Hamiltonian**:
$$\hat{H}_C = \sum_{(i,j) \in E} \frac{1}{2} Z_i Z_j$$

Minimizing the expectation value $\langle \hat{H}_C \rangle$ corresponds directly to maximizing the number of cut edges!


In [ ]:
# Define a small 5-node introductory graph
toy_edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]
num_nodes = 5

def build_maxcut_hamiltonian(nodes: int, edges: list) -> SparsePauliOp:
    """Builds the Max-Cut cost Hamiltonian sum_(i,j) 0.5 * Z_i Z_j."""
    terms = []
    for u, v in edges:
        pauli = ["I"] * nodes
        pauli[u] = "Z"
        pauli[v] = "Z"
        terms.append(("".join(pauli), 0.5))
    return SparsePauliOp.from_list(terms)

cost_hamiltonian = build_maxcut_hamiltonian(num_nodes, toy_edges)
print(f"Constructed Cost Hamiltonian on {num_nodes} nodes with {len(cost_hamiltonian)} Pauli terms:")
print(cost_hamiltonian)


## Part 2: Building the Parameterized QAOA Circuit

A depth-$p$ QAOA circuit applies alternating unitary transformations:
1. **Initial Superposition:** $|\psi_0\rangle = H^{\otimes n} |0\rangle = |+\rangle^{\otimes n}$.
2. **Cost Layer:** $U(C, \gamma) = e^{-i \gamma \hat{H}_C} = \prod_{(i,j) \in E} e^{-i \frac{\gamma}{2} Z_i Z_j}$.
3. **Mixer Layer:** $U(B, \beta) = e^{-i \beta \hat{H}_M} = \prod_{i \in V} e^{-i \beta X_i}$.

The state is parameterized by $2p$ angles:
$$|\psi(\vec{\gamma}, \vec{\beta})\rangle = U(B, \beta_p) U(C, \gamma_p) \dots U(B, \beta_1) U(C, \gamma_1) |+\rangle^{\otimes n}$$


In [ ]:
# QAOA with depth p = 1
p_steps = 1
qaoa_ansatz = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=p_steps)

estimator = StatevectorEstimator()
optimization_history = []

def qaoa_cost_function(params):
    job = estimator.run([(qaoa_ansatz, cost_hamiltonian, params)])
    energy = float(job.result()[0].data.evs)
    optimization_history.append(energy)
    return energy

# Classical optimization with COBYLA
initial_angles = [0.5, 0.5] * p_steps
result = minimize(qaoa_cost_function, initial_angles, method='COBYLA')

optimal_angles = result.x
optimal_energy = float(result.fun)

print(f"[✓] QAOA Optimization Completed:")
print(f"    Optimal Angles (gamma, beta): {optimal_angles}")
print(f"    Minimum Energy <H_C>:          {optimal_energy:.4f}")

# Plot convergence
plt.figure(figsize=(7, 3.5))
plt.plot(optimization_history, color='#4C72B0', lw=2)
plt.xlabel("Iteration")
plt.ylabel(r"Expected Cost $\langle \hat{H}_C \rangle$")
plt.title("QAOA Parameter Convergence")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Part 3: Sampling the Solutions & Computing the Approximation Ratio

After finding the optimal variational angles $(\vec{\gamma}^*, \vec{\beta}^*)$, we bind them into the measured circuit and sample the output bitstrings using `StatevectorSampler`.

We compare QAOA's cut against the exact brute-force optimum to compute the **Approximation Ratio ($\alpha$)**:
$$\alpha = \frac{C_{\text{qaoa}}}{C_{\text{optimal}}}$$


In [ ]:
def evaluate_cut(bitstring: str, edges: list) -> int:
    """Computes the number of cut edges for a binary partition."""
    return sum(1 for u, v in edges if bitstring[u] != bitstring[v])

# 1. Classical brute-force optimum
max_possible_cut = 0
for i in range(2**num_nodes):
    bits = f"{i:0{num_nodes}b}"
    cut = evaluate_cut(bits, toy_edges)
    if cut > max_possible_cut:
        max_possible_cut = cut

# 2. Sample from the optimized quantum state
sampler = StatevectorSampler()
measured_qaoa = qaoa_ansatz.assign_parameters(optimal_angles)
measured_qaoa.measure_all()

sample_counts = sampler.run([measured_qaoa], shots=2000).result()[0].data.meas.get_counts()

# Find the best cut sampled by QAOA
qaoa_best_cut = max(evaluate_cut(b, toy_edges) for b in sample_counts.keys())
approximation_ratio = qaoa_best_cut / max_possible_cut

print(f"Brute-Force Optimal Cut: {max_possible_cut} edges")
print(f"QAOA Sampled Best Cut:   {qaoa_best_cut} edges")
print(f"Approximation Ratio (alpha): {approximation_ratio:.2%}")


---

## Part 4: Scaling to 156 Qubits on Native IBM Heavy-Hex Hardware

On real quantum processors (such as the 156-qubit **IBM Heron** architecture), arbitrary graphs require deep SWAP networks that degrade quantum coherence.

However, if we embed our problem graph as a **subgraph of the native heavy-hex coupling map**, every interaction edge $(i, j)$ directly connects two physical qubits on the chip:
$$\text{Native Coupling Map} \implies \text{Zero SWAP Gates!}$$

Let's generate a scalable hardware-native graph on **$N = 12, 24, 64,$ or $156$ qubits** and transpile it onto the native coupling map to prove that $N_{\text{swap}} = 0$:


In [ ]:
def make_heavy_hex_subgraph(nodes: int) -> tuple[int, list]:
    """Generates a hardware-native subgraph matching IBM Heavy-Hex connectivity (degree <= 3)."""
    edges = []
    # Primary linear backbone
    for i in range(nodes - 1):
        edges.append((i, i + 1))
    # Periodic cross-hexagonal bridges (degree <= 3)
    for i in range(0, nodes - 4, 4):
        edges.append((i, i + 3))
    return nodes, edges

# Choose graph scale: 12, 24, 64, or up to 156 qubits!
SCALE_QUBITS = 12  # Try scaling: 24, 64, or 156!
n_qubits, heavy_hex_edges = make_heavy_hex_subgraph(SCALE_QUBITS)

# 1. Build hardware coupling map from the graph
hardware_cmap = CouplingMap(heavy_hex_edges + [(v, u) for u, v in heavy_hex_edges])

# 2. Build cost Hamiltonian and QAOA circuit
scale_hamiltonian = build_maxcut_hamiltonian(n_qubits, heavy_hex_edges)
scale_qaoa = QAOAAnsatz(cost_operator=scale_hamiltonian, reps=1)
assigned_scale = scale_qaoa.assign_parameters([0.4, 0.4]).decompose()

# 3. Transpile directly onto the native hardware coupling map
transpiled_hardware_qaoa = transpile(
    assigned_scale,
    coupling_map=hardware_cmap,
    optimization_level=3,
    seed_transpiler=42
)

hw_ops = dict(transpiled_hardware_qaoa.count_ops())
hw_swaps = hw_ops.get('swap', 0)

print(f"Transpiled Native Heavy-Hex QAOA Circuit ({n_qubits} Qubits):")
print(f"  Circuit Gate Depth: {transpiled_hardware_qaoa.depth()}")
print(f"  SWAP Gate Count:    {hw_swaps} (Achieves 0 SWAPs on native hardware!)")
print(f"  Native Operations:  {hw_ops}")


## Part 5: Local Score Previewer (Test Before Submitting)

This helper function mirrors the official backend grading algorithm for `challenge_advanced`. It evaluates:
- **Unbounded base potential** scaling as $1,000 \times N^{2.5}$
- **Approximation factor** $(\alpha)$
- **SWAP penalty** ($N_{\text{swap}}$)
- **Depth efficiency** ($D$)


In [ ]:
def preview_qaoa_score(circuit: QuantumCircuit, approx_ratio: float):
    """Calculates the projected score for the advanced QAOA challenge."""
    num_qubits = circuit.num_qubits
    ops = dict(circuit.count_ops())
    gate_depth = circuit.depth()
    swap_count = ops.get('swap', 0)
    
    if num_qubits < 5:
        print(f"[!] REJECTED: Circuit has {num_qubits} qubits. Advanced challenge requires at least 5 qubits.")
        return 0
    if approx_ratio <= 0.50:
        print(f"[!] Approximation ratio ({approx_ratio:.2%}) is <= random guess (50%). Score = 0.")
        return 0
        
    # 1. Base Score: 5000 + 60 * (N^2 - 25)
    base_score = 5000.0 + 60.0 * (num_qubits ** 2 - 25)
    
    # 2. Approximation Factor
    norm_approx = (min(1.0, approx_ratio) - 0.50) / 0.50
    approx_factor = norm_approx ** 2
    
    # 3. SWAP Penalty
    swap_penalty = 1.0 / (1.0 + 0.20 * swap_count)
    
    # 4. Depth Efficiency (reference hardware-native depth = 6)
    ref_depth = 6
    depth_efficiency = ref_depth / max(gate_depth, ref_depth)
    
    total_score = round(base_score * approx_factor * swap_penalty * depth_efficiency)
    
    print("=" * 50)
    print(f"ADVANCED QAOA SCORE BREAKDOWN ({num_qubits} Qubits)")
    print("=" * 50)
    print(f"  Base Potential:       {base_score:,.0f} pts")
    print(f"  Approximation Ratio:  {approx_ratio:.2%} (Factor: {approx_factor:.4f})")
    print(f"  SWAP Gates:           {swap_count} (Penalty: {swap_penalty:.3f})")
    print(f"  Circuit Gate Depth:   {gate_depth} (Ref: {ref_depth}, Efficiency: {depth_efficiency:.3f})")
    print("-" * 50)
    print(f"  PROJECTED SCORE:      {total_score:,} POINTS")
    print("=" * 50)
    return total_score

# Compute preview for our native heavy-hex circuit (assuming 90% approximation ratio)
simulated_approx_ratio = 0.90
projected_score = preview_qaoa_score(transpiled_hardware_qaoa, simulated_approx_ratio)


## Part 6: Submit to the Platform

Submit your optimal QAOA circuit QASM and the achieved approximation ratio to the live hackathon platform.
- Target `challenge_advanced`.
- Make sure to use your team API key from your Participant Dashboard.
- Watch your team climb the live leaderboard!


In [ ]:
# Configuration
API_KEY = "YOUR_TEAM_API_KEY_HERE"  # Replace with your team API key
SUBMISSION_URL = "http://localhost:3000/api/submissions"
CHALLENGE_ID = "challenge_advanced"

payload = {
    "qasm": qasm2.dumps(transpiled_hardware_qaoa),
    "challengeId": CHALLENGE_ID,
    "obtainedResult": float(simulated_approx_ratio)
}

print(f"Submitting {transpiled_hardware_qaoa.num_qubits}-qubit QAOA solution to {CHALLENGE_ID}...")
try:
    response = requests.post(
        SUBMISSION_URL,
        json=payload,
        headers={"X-API-Key": API_KEY},
        timeout=20
    )
    if response.ok:
        data = response.json()
        print("\n[✓] Submission successful!")
        print(f"    Submission ID: {data['submission']['id']}")
        print(f"    Official Score: {data['submission']['score']:,} pts")
    else:
        print(f"\n[X] Submission failed ({response.status_code}):")
        print("   ", response.json().get("error", response.text))
except Exception as e:
    print(f"\n[!] Network or request error: {e}")
